In [4]:
import pandas as pd
import networkx as nx
import json
import re
df = pd.read_parquet("hf://datasets/notesbymuneeb/epstein-emails/epstein_email_threads.parquet")

In [13]:
df.head()

,thread_id,source_file,subject,messages,message_count
0,TEXT-001-HOUSE_OVERSIGHT_031683.txt_2,TEXT-001-HOUSE_OVERSIGHT_031683.txt,Re:,"[{""sender"": ""J [jeevacation@gmail.com]"", ""reci...",3
1,TEXT-001-HOUSE_OVERSIGHT_022827.txt_12,TEXT-001-HOUSE_OVERSIGHT_022827.txt,Re:,"[{""sender"": ""jeffrey E. <jeevacation@gmail.com...",5
2,TEXT-001-HOUSE_OVERSIGHT_025190.txt_14,TEXT-001-HOUSE_OVERSIGHT_025190.txt,Kuhn - The Science of Sleep and Dreams,"[{""sender"": ""Robert Lawrence Kuhn"", ""recipient...",7
3,TEXT-001-HOUSE_OVERSIGHT_026499.txt_18,TEXT-001-HOUSE_OVERSIGHT_026499.txt,Fwd: Our May conversation - more thoughts,"[{""sender"": ""jeffrey E. <jeevacation@gmail.com...",1
4,TEXT-002-HOUSE_OVERSIGHT_032654.txt_19,TEXT-002-HOUSE_OVERSIGHT_032654.txt,Re: Marilyn,"[{""sender"": ""jeffrey E. <jeevacation@gmail.com...",8


In [5]:
df['messages']

0       [{"sender": "J [jeevacation@gmail.com]", "reci...
1       [{"sender": "jeffrey E. <jeevacation@gmail.com...
2       [{"sender": "Robert Lawrence Kuhn", "recipient...
3       [{"sender": "jeffrey E. <jeevacation@gmail.com...
4       [{"sender": "jeffrey E. <jeevacation@gmail.com...
                              ...                        
5077    [{"sender": "Lesley Groff", "recipients": ["Je...
5078    [{"sender": "Lawrence Krauss", "recipients": [...
5079    [{"sender": "Richard Kahn", "recipients": ["Je...
5080    [{"sender": "Richard Kahn", "recipients": ["je...
5081    [{"sender": "Jeffrey Epstein <jeevacation@gmai...
Name: messages, Length: 5082, dtype: str

In [6]:
g=nx.Graph()

In [7]:
for index, row in df.iterrows():
    messages = json.loads(row['messages'])
    for message in messages:
        if message['sender']:
            g.add_node(message['sender'])
            if message['recipients']:
                for recipient in message['recipients']:
                    g.add_node(recipient)
                    current_weight = g[message['sender']][recipient]['weight'] if g.has_edge(message['sender'], recipient) else 0
                    g.add_edge(recipient, message['sender'],weight=current_weight+1)

print('Nodes:', len(g.nodes))
print('Edges:', len(g.edges))

Nodes: 2830
Edges: 3388


In [6]:
centrality_degree = nx.degree_centrality(g)
top_k = 20
print("\n== Top 20 by degree centrality===")
for u in sorted(centrality_degree, key=centrality_degree.get,reverse=True)[:top_k]:
    print(u,centrality_degree[u])


== Top 20 by degree centrality===
jeffrey E. [jeevacation@gmail.com] 0.16401555319901026
jeffrey E. <jeevacation@gmail.com> 0.13184870979144575
Jeffrey Epstein [jeevacation@gmail.com] 0.07246376811594203
Jeffrey Epstein <jeevacation@gmail.com> 0.0607988688582538
jeevacation@gmail.com 0.04736656062212796
J <jeevacation@gmail.com> 0.043478260869565216
J [jeevacation@gmail.com] 0.036055143160127257
Terry Kafka 0.02509720749381407
ee 0.02156238953693885
Darren Indyke 0.020501944149876283
Robert Trivers 0.019088016967126194
Jeffrey Epstein 0.017320607988688584
Weingarten, Reid 0.016260162601626018
Lesley Groff 0.014846235418875928
Jeffrey E. <jeevacation@gmail.com> 0.014846235418875928
Jeffrey E. [jeevacation@gmail.com] 0.012371862849063274
jeffrey E. 0.012018381053375752
Richard Kahn 0.011311417462000707
Martin Weinberg 0.010957935666313185
Keough, Michael 0.010957935666313185


In [7]:
centrality_degree = nx.betweenness_centrality(g)
top_j = 20
print("\n== Top 20 by betweenness centrality===")
for u in sorted(centrality_degree, key=centrality_degree.get,reverse=True)[:top_j]:
    print(u,centrality_degree[u])


== Top 20 by betweenness centrality===
jeffrey E. [jeevacation@gmail.com] 0.21489094670656267
jeffrey E. <jeevacation@gmail.com> 0.1730699240279016
Jeffrey Epstein [jeevacation@gmail.com] 0.08846287484179406
jeevacation@gmail.com 0.08067331604221992
Jeffrey Epstein <jeevacation@gmail.com> 0.07140185238053387
J <jeevacation@gmail.com> 0.05206726403790867
Terry Kafka 0.0514363220867098
Weingarten, Reid 0.0373982817675634
J [jeevacation@gmail.com] 0.0358439452142508
ee 0.03399156180506496
Darren Indyke 0.03310657144231863
paul krassner 0.030534086361287995
Robert Trivers 0.026591069012629392
Martin Weinberg 0.024458287213357188
Steve Bannon 0.015245662971924959
Lesley Groff 0.013839678510098478
Lawrence Krauss 0.013472083588170978
Tonja Haddad Coleman 0.012561997568246977
Jeffrey Epstein 0.012062363582313086
Boris Nikolic 0.011235960745375455


In [8]:
def get_email(text):
    match = re.search(r'[\w\.-]+@[\w\.-]+', text); 
    return match.group(0).lower() if match else text.lower()

In [9]:
t = nx.Graph()

In [10]:
for index, row in df.iterrows():
    messages = json.loads(row['messages'])
    for message in messages:
        if message['sender']:
            sender = get_email(message['sender'])
            if sender:
                t.add_node(sender)
                if message['recipients']:
                    for recipient in message['recipients']:
                        r = get_email(recipient)
                        if r:
                            t.add_node(r)
                            current_weight = t[sender][r]['weight'] if t.has_edge(sender, r) else 0
                            t.add_edge(sender, r, weight=current_weight+1)

print('Nodes:', len(t.nodes))
print('Edges:', len(t.edges))

Nodes: 2577
Edges: 2684


In [11]:
centrality_degree = nx.degree_centrality(t)
top_d = 20
print("\n== Top 20 by degree centrality===")
for u in sorted(centrality_degree, key=centrality_degree.get,reverse=True)[:top_d]:
    print(u,centrality_degree[u])


== Top 20 by degree centrality===
jeevacation@gmail.com 0.4658385093167702
terry kafka 0.019798136645962732
robert trivers 0.019409937888198756
darren indyke 0.018633540372670808
jeffrey epstein 0.018633540372670808
ee 0.018245341614906832
jeffrey e. 0.01436335403726708
lesley groff 0.013198757763975154
jeffrey 0.013198757763975154
keough, michael 0.012034161490683228
weingarten, reid 0.010093167701863354
paul krassner 0.009704968944099378
lawrence krauss 0.009316770186335404
richard kahn 0.008928571428571428
paulkrassner@roadrunner.com 0.008928571428571428
martin weinberg 0.008540372670807452
martin g. weinberg 0.006599378881987577
king, kathryn 0.006599378881987577
will bohlen 0.006599378881987577
pizzurro, frank (la) 0.006599378881987577


In [12]:
centrality_degree = nx.betweenness_centrality(t)
top_e = 20
print("\n== Top 20 by betweenness centrality===")
for u in sorted(centrality_degree, key=centrality_degree.get,reverse=True)[:top_e]:
    print(u,centrality_degree[u])


== Top 20 by betweenness centrality===
jeevacation@gmail.com 0.5814380402208321
terry kafka 0.04450540333152468
robert trivers 0.02860519809443406
paul krassner 0.02341452709484732
ee 0.02148842698979697
weingarten, reid 0.021090869579063257
lawrence krauss 0.01833139778354005
darren indyke 0.015695310939326034
martin weinberg 0.015363903660761341
keough, michael 0.011975575028206916
brad bowen 0.011396912500753784
jay lefkowitz 0.011311335173633903
steve bannon 0.010508527809604212
tom 0.010213228045480947
miller, michael 0.010203971373145724
tonja haddad coleman 0.010202865538683846
hill, james e. 0.010186636917324972
paulkrassner@roadrunner.com 0.009926007562291707
lesley groff 0.008828570431281551
martin g. weinberg 0.008782908081625577


In [14]:
nx.write_graphml(t, "epsteins.graphml")
print('GraphMl exported to actors.graphml')

GraphMl exported to actors.graphml
